# 7주차 실습 — SQL (1) 조회 기초

- **데이터**: `aact_mini` — 임상시험 등록 자료(ClinicalTrials.gov)에서 추린 수업용 데이터베이스입니다. 오늘은 `studies`(임상시험 1건 = 1행, 37,509행) 한 표만 씁니다.
- **오늘 배우는 것**
  1. 실습 환경에서 SQL을 실행할 수 있다
  2. `SELECT` · `FROM`으로 열을 고를 수 있다
  3. `WHERE`로 필요한 행만 고를 수 있다
  4. 오류 메시지를 읽고 고칠 수 있다
- **사용법**: 따라 하기(1~4절) → 자주 하는 실수(5절) → 직접 해보기 8문항. 각 문항 아래 `check("Q1", _)` 를 실행하면 정답 여부가 바로 표시됩니다.
- **점수는 없습니다.** 끝까지 해 보는 것이 목표입니다.

## 0. 시작하기 전에

- **Colab 에서 열었다면 먼저 메뉴의 `파일 → Drive에 사본 저장`** 을 누르세요. 그래야 작성한 내용이 남습니다.
- 셀 실행: 셀 왼쪽의 ▶ 단추 또는 `Shift + Enter`. **위에서부터 차례로** 실행합니다.
- `%%sql` 로 시작하는 셀에는 SQL만 적습니다. `%%sql` 바로 아래의 `-- Q1` 같은 줄은 지우지 마세요 (문항 이름표입니다).
- 오류가 나면 붉은 글씨 중 **`Error:` 가 들어간 줄**만 읽으면 됩니다. 연결이 이상하면 `런타임 → 런타임 다시 시작` 후 준비 셀부터 다시 실행하세요.

In [ ]:
#@title ▶ 준비 셀 — 실행만 하세요 (Shift+Enter). 코드는 몰라도 됩니다. { display-mode: "form" }
import os, sys, urllib.request
import numpy as _np, pandas as _pd
DB  = "aact_mini.duckdb"
URL = "https://raw.githubusercontent.com/kevinkwshin/medical-db-course-2026/main/data/aact_mini.duckdb.gz"

# 로컬(수업 폴더)에서 열었을 때는 ../data/ 의 파일을 사용
for cand in [DB, os.path.join("..", "data", DB), os.path.join("data", DB)]:
    if os.path.exists(cand):
        DB = cand
        break
else:
    print("데이터 파일을 내려받습니다 (약 50MB, 1분 내외) ...")
    import gzip, shutil
    urllib.request.urlretrieve(URL, DB + ".gz")
    with gzip.open(DB + ".gz", "rb") as f_in, open(DB, "wb") as f_out:
        shutil.copyfileobj(f_in, f_out)
    os.remove(DB + ".gz")

if "google.colab" in sys.modules:           # Colab 에서만 설치 (수업에서 검증한 버전)
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb==1.4.5", "jupysql==0.11.1", "duckdb-engine==0.17.0"])

import duckdb
_pd.set_option("display.max_colwidth", 60)
%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.displaylimit = 30
%config SqlMagic.feedback = 0
%xmode Minimal
try:                                         # jupysql 의 영어 안내문(If using snippets ...)을 숨김
    import sql.error_handler as _eh
    _eh.CTE_MSG, _eh.ORIGINAL_ERROR = "", ""
except Exception:
    pass
conn = duckdb.connect(DB, read_only=True)
conn.execute("SET python_enable_replacements = false")   # 빈칸 ___ 을 파이썬 변수로 읽지 않게
%sql conn --alias aact

import hashlib as _hl, math as _math, decimal as _dec, datetime as _dt

def _v(v):
    """값 하나를 비교용 문자열로: 빈 값 → '', 1947.0 → '1947', 날짜 → 'YYYY-MM-DD'"""
    try:
        if v is None or (not isinstance(v, (str, bytes)) and _pd.isna(v)):
            return ""
    except (TypeError, ValueError):
        pass
    if isinstance(v, _dec.Decimal):
        v = float(v)
    if isinstance(v, (bool, _np.bool_)):
        return str(bool(v))
    if isinstance(v, (int, _np.integer)):
        return str(int(v))
    if isinstance(v, (float, _np.floating)):
        v = round(float(v), 4)
        return str(int(v)) if v.is_integer() else repr(v)
    if isinstance(v, (_pd.Timestamp, _dt.datetime)):
        if v.hour == 0 and v.minute == 0 and v.second == 0 and v.microsecond == 0:
            return v.strftime("%Y-%m-%d")
        return v.isoformat()
    if isinstance(v, _dt.date):
        return v.isoformat()
    return str(v)

def _sigs(df):
    """결과표의 지문: 열 이름(소문자), 행 수, 행 순서 무시 지문(sig), 행 순서 포함 지문(osig)"""
    cols = [str(c).strip().lower() for c in df.columns]
    rows = ["\x1f".join(_v(x) for x in r) for r in df.itertuples(index=False, name=None)]
    head = "\x1e".join(cols)
    h = lambda rs: _hl.md5((head + "\x1d" + "\x1e".join(rs)).encode()).hexdigest()[:12]
    return {"sig": h(sorted(rows)), "osig": h(rows), "rows": len(rows), "cols": cols}

_LAST = {"tag": None, "error": False}
_TRIES, _RESULT = {}, {}

def _after_cell(result):
    info = getattr(result, "info", None)
    src = (getattr(info, "raw_cell", "") or "").lstrip()
    if not src.startswith("%%sql"):
        return
    tag = None
    for line in src.split("\n")[1:]:
        s = line.strip()
        if s.startswith("--"):
            parts = s[2:].split()
            tag = parts[0] if parts and not parts[0].startswith("[") else None
            break
        if s:
            break
    _LAST.update(tag=tag, error=not result.success)

for _cb in list(get_ipython().events.callbacks["post_run_cell"]):
    if getattr(_cb, "__name__", "") == "_after_cell":
        get_ipython().events.unregister("post_run_cell", _cb)
get_ipython().events.register("post_run_cell", _after_cell)

def _expected(name):
    a = ANSWERS[name]
    if "by_mesh" in a:
        if not globals().get("MY_MESH"):
            return None
        return dict(a["by_mesh"][MY_MESH], hint=a.get("hint", ""), ordered=a.get("ordered", False))
    return a

def _miss(name, why, exp):
    """틀렸을 때: 기록하고, 두 번째부터 힌트를 붙여 출력"""
    _RESULT[name] = False
    _TRIES[name] = _TRIES.get(name, 0) + 1
    msg = f"❌ {name}  {why}"
    hint = (exp or ANSWERS.get(name, {})).get("hint")
    if hint:
        msg += ("\n   💡 힌트: " + hint) if _TRIES[name] >= 2 else "\n   (한 번 더 다르면 힌트가 나옵니다)"
    print(msg)

def check(name, df=None):
    """바로 위 %%sql 셀의 결과를 확인합니다.  예) check("Q1", _)"""
    if name not in ANSWERS:
        print(f"알 수 없는 문항 이름입니다: {name}"); return
    if _LAST["tag"] and _LAST["tag"] != name:
        print(f"⚠️ 방금 실행한 SQL 셀은 {_LAST['tag']} 입니다. {name} 의 %%sql 셀을 먼저 실행한 뒤 이 셀을 실행하세요."); return
    exp = _expected(name)
    if exp is None:
        print("⚠️ 학번이 입력되지 않아 배정 질환이 없습니다. 맨 위 학번 셀과 준비 셀을 다시 실행하세요."); return
    if _LAST["error"]:
        return _miss(name, "바로 위 SQL 셀에서 오류가 났습니다. 붉은 글씨 중 'Error:' 가 들어간 줄을 읽고 SQL을 고친 뒤 다시 실행하세요.", exp)
    if not isinstance(df, _pd.DataFrame):
        return _miss(name, f"결과가 표가 아닙니다. %%sql 셀을 실행한 뒤, 바로 다음 셀에서 check('{name}', _) 를 실행하세요.", exp)
    got = _sigs(df)
    if got["sig"] == exp["sig"] and (not exp.get("ordered") or got["osig"] == exp["osig"]):
        _RESULT[name] = True
        print(f"✅ {name} 정답!  (행 {got['rows']}개)"); return
    zero = "값의 철자 · 대소문자 · 따옴표(예: 'COMPLETED')와, 빈 값을 = 나 <> 로 비교하지 않았는지(IS NULL) 확인하세요."
    if got["sig"] == exp["sig"]:
        why = "값은 맞지만 행의 순서가 다릅니다. 정렬(ORDER BY · DESC)을 확인하세요."
    elif got["cols"] != exp["cols"]:
        if len(got["cols"]) != len(exp["cols"]):
            why = f"열 개수가 다릅니다 (기대 {len(exp['cols'])}개: {', '.join(exp['cols'])} / 현재 {len(got['cols'])}개)."
        elif sorted(got["cols"]) == sorted(exp["cols"]):
            why = f"열의 순서가 다릅니다. 문항에 적힌 순서대로 적으세요: {', '.join(exp['cols'])}"
        else:
            why = f"열 이름이 다릅니다. 기대: {', '.join(exp['cols'])}  /  현재: {', '.join(got['cols'])}  (AS 로 이름을 맞추세요)"
    elif got["rows"] == 0 and exp["rows"] > 0:
        why = "결과가 0행입니다. " + zero
    elif got["rows"] == 1 and len(df.columns) and all(_v(x) in ("0", "") for x in df.iloc[0]):
        why = "결과가 0 이거나 비어 있습니다. 조건에 맞는 행이 없다는 뜻입니다. " + zero
    elif got["rows"] != exp["rows"]:
        why = f"행 수가 다릅니다 (기대 {exp['rows']}행, 현재 {got['rows']}행). 조건과 LIMIT 을 확인하세요."
    elif exp.get("ordered"):
        why = "행 수는 맞지만 값이나 순서가 다릅니다. 조건과 정렬(ORDER BY · DESC)을 확인하세요."
    else:
        why = "행 수는 맞지만 값이 다릅니다. 조건이나 계산식을 확인하세요."
    _miss(name, why, exp)

def 진행확인():
    """지금까지 확인한 문항의 결과를 한눈에 봅니다. 선택 문항은 개수에서 뺍니다."""
    names = list(ANSWERS)
    core = [n for n in names if not ANSWERS[n].get("optional")]
    ok = [n for n in core if _RESULT.get(n) is True]
    print(f"정답 {len(ok)} / {len(core)} 문항")
    for n in names:
        mark = {True: "✅", False: "❌"}.get(_RESULT.get(n), "·  (아직 check 안 함)")
        print(f"  {n:>4}  {mark}" + ("   (선택)" if ANSWERS[n].get("optional") else ""))

print("연결 완료:", DB)

In [ ]:
#@title 채점용 정보 (수정하지 마세요) { display-mode: "form" }
ANSWERS = {'Q1': {'sig': '63b876ca435e', 'osig': '523478138cd2', 'rows': 37509, 'cols': ['등록번호', '진행상태'], 'hint': 'SELECT 뒤에 볼 열 이름을 문항에 적힌 순서대로 적습니다. 모든 시험이 나와야 하므로 LIMIT 은 붙이지 않습니다.', 'ordered': False, 'optional': False}, 'Q2': {'sig': '44720f8bc3b6', 'osig': '44720f8bc3b6', 'rows': 1, 'cols': ['중단시험'], 'hint': '문자 값은 작은따옴표로 감싸고, 대문자 그대로 적습니다.', 'ordered': False, 'optional': False}, 'Q3': {'sig': '63a416a66ad7', 'osig': '63a416a66ad7', 'rows': 1, 'cols': ['시험수'], 'hint': '두 조건을 모두 만족해야 하면 AND 입니다. ‘이상’은 그 값도 포함합니다. 숫자는 따옴표 없이 씁니다.', 'ordered': False, 'optional': False}, 'Q4': {'sig': '39fb93d82d56', 'osig': '39fb93d82d56', 'rows': 1, 'cols': ['시험수'], 'hint': "BETWEEN 은 양 끝을 포함합니다. 날짜는 'YYYY-MM-DD' 형태로 작은따옴표 안에 적습니다.", 'ordered': False, 'optional': False}, 'Q5': {'sig': 'ee79bd6e9b5f', 'osig': 'ee79bd6e9b5f', 'rows': 1, 'cols': ['시험수'], 'hint': '제목을 대문자로 바꿨으니 찾는 글자도 대문자로 적고, 앞뒤에 % 를 붙여 작은따옴표로 감쌉니다.', 'ordered': False, 'optional': False}, 'Q6': {'sig': 'b060ebda25ea', 'osig': 'b060ebda25ea', 'rows': 1, 'cols': ['결과있음'], 'hint': '빈 값은 = 나 <> 로 비교할 수 없습니다. ‘값이 있다’는 IS NULL 의 반대입니다.', 'ordered': False, 'optional': False}, 'Q7': {'sig': '5a12b3898bd0', 'osig': '9d19b26abfec', 'rows': 5, 'cols': ['nct_id', 'enrollment'], 'hint': '큰 값부터 정렬하려면 DESC, 앞에서 몇 줄만 보려면 LIMIT 입니다.', 'ordered': True, 'optional': False}, 'Q8': {'sig': 'd571b6b2402e', 'osig': '92e593093552', 'rows': 9, 'cols': ['phase'], 'hint': '중복을 없애고 값의 종류만 보려면 SELECT 바로 뒤에 한 단어를 넣습니다.', 'ordered': False, 'optional': False}}

## 1. 실습 환경 — Colab과 aact_mini

> **병원 데이터라면** — 평소 진료에서는 EMR 프로그램이 SQL을 대신 만들어 줍니다. 그런데 “작년에 65세 이상 당뇨 환자 중 재입원한 사람” 같은 질문은 화면에 버튼이 없습니다. 직접 SQL로 물어봐야 합니다.

- 위의 준비 셀이 데이터 파일을 받아 연결했습니다. 데이터는 **읽기 전용**으로 엽니다. 실수로 지우거나 바꿀 수 없습니다.
- 실습 표는 미리 만들어 두었습니다. 표를 만드는 `CREATE TABLE`(DDL)과 권한을 주는 `GRANT`(DCL)는 슬라이드 165~168에서 읽기만 합니다.
- **DuckDB** — 파일 하나로 된 관계형 DBMS입니다. 문법은 PostgreSQL과 거의 같습니다.
- **jupysql** — `%%sql` 로 시작하는 칸에 SQL만 입력하면 결과가 표로 바로 나옵니다.

먼저 어떤 표가 있는지 봅니다. `information_schema.tables` 는 표 목록을 보는 표준 방법(데이터 사전)입니다. 표 이름 14개가 나오면 연결된 것입니다. (SQL 문장은 2절에서 배웁니다. 지금은 실행만 하세요.)

In [ ]:
%%sql
-- 어떤 표(테이블)가 있는지
SELECT table_name
FROM information_schema.tables;

수업에서 쓰는 주요 표입니다 (슬라이드 172).

| 테이블 | 한 행이 뜻하는 것 | 행 수 | 주로 쓰는 열 |
|---|---|---:|---|
| `studies` | 임상시험 1건 | 37,509 | nct_id, brief_title, start_date, overall_status, phase, enrollment, results_first_posted_date |
| `conditions` | 대상 질환 1개 | 73,173 | nct_id, name |
| `eligibilities` | 선정 · 제외 기준 | 37,509 | nct_id, gender, minimum_age |
| `sponsors` | 스폰서 1곳 | 62,783 | nct_id, agency_class, name |
| `facilities` | 수행 기관 1곳 | 208,690 | nct_id, name, country |
| `browse_conditions` | 표준 질환명 1개 | 309,124 | nct_id, mesh_term |
| `countries` | 참여 국가 1곳 | 50,410 | nct_id, name, removed |

오늘은 **studies** 한 표만 씁니다. 여러 표를 연결하는 것은 8주차 JOIN에서 배웁니다.

## 2. SELECT · FROM — 필요한 열 고르기

SQL 문장은 다섯 부분을 **순서대로** 적습니다. 순서를 바꿔 쓰면 문법 오류가 납니다.

| 절 | 뜻 | 예 |
|---|---|---|
| `SELECT` | 무엇을 볼지 (열) | `nct_id, brief_title` |
| `FROM` | 어디에서 (테이블) | `studies` |
| `WHERE` | 어떤 행만 (조건) | `phase = 'PHASE3'` |
| `ORDER BY` | 어떤 순서로 (정렬) | `start_date DESC` |
| `LIMIT` | 몇 줄만 (개수) | `10` |

**SELECT → FROM → WHERE → ORDER BY → LIMIT**

**SELECT · FROM** — 가장 기본이 되는 두 줄입니다. `LIMIT 3` 은 앞에서 3줄만 보라는 뜻입니다. 큰 표를 처음 볼 때 꼭 붙입니다.
(정렬 없이 앞의 3줄을 보여 주므로, 이 셀과 아래 AS 예제는 슬라이드와 다른 행이 나올 수 있습니다.)

In [ ]:
%%sql
-- “임상시험의 등록번호와 제목을 보여 줘”
SELECT nct_id, brief_title
FROM studies
LIMIT 3;

**전체 열 보기와 개수 세기** — 별표(`*`)는 모든 열, `COUNT(*)`는 행 수입니다.
`SELECT *` 는 열이 30개나 나오니 필요한 열만 적으세요. COUNT는 8주차에 자세히 배웁니다.

In [ ]:
%%sql
-- “이 표에 행이 몇 개나 있어?”  → 37,509
SELECT COUNT(*) AS 시험수
FROM studies;

**AS — 결과 열 이름 바꾸기** — 보고서에 그대로 쓸 수 있게 이름을 붙입니다. `AS` 는 생략할 수 있지만, 적어 두면 나중에 읽기 훨씬 좋습니다.

In [ ]:
%%sql
-- “등록번호와 등록인원을 한글 이름으로 보여 줘”
SELECT nct_id AS 등록번호,
       enrollment AS 등록인원
FROM studies
LIMIT 3;

## 3. WHERE — 필요한 행 고르기

조건은 이 여섯 가지 모양이면 거의 다 됩니다.

| 종류 | 쓰는 법 | 예 | 뜻 |
|---|---|---|---|
| 비교 | `=` `<>` `>` `>=` `<` `<=` (`<>` 는 ‘같지 않다’) | `enrollment >= 1000` | 등록인원 1000명 이상 |
| 여러 조건 | `AND` `OR` `NOT` | `A AND B` | 두 조건을 모두 만족 |
| 목록 | `IN (값1, 값2)` | `phase IN ('PHASE3','PHASE4')` | 여러 값 중 하나 |
| 범위 | `BETWEEN a AND b` | `start_date BETWEEN '2024-01-01' AND '2024-12-31'` | 양 끝을 포함한 범위 |
| 글자 | `LIKE '%글자%'` | `brief_title LIKE '%Nurs%'` | 그 글자가 들어간 것 |
| 빈 값 | `IS NULL` / `IS NOT NULL` | `results_first_posted_date IS NULL` | 값이 없는 / 있는 행 |

문자와 날짜는 반드시 **'작은따옴표'** 로 감쌉니다. 숫자는 따옴표 없이 씁니다.

**WHERE — 조건 하나** — 전체 37,509건 중 19,622건입니다. 문자 값 `'COMPLETED'` 는 작은따옴표로 감쌉니다.

In [ ]:
%%sql
-- “완료(COMPLETED)된 시험이 몇 건이야?”
SELECT COUNT(*) AS 완료시험
FROM studies
WHERE overall_status = 'COMPLETED';

**WHERE — 조건 두 개 (AND)** — 조건을 하나씩 추가하면서 건수가 어떻게 줄어드는지 확인하는 습관이 중요합니다.

In [ ]:
%%sql
-- “2024년에 시작한 3상 시험을 보여 줘”
SELECT nct_id, start_date, phase
FROM studies
WHERE start_date BETWEEN '2024-01-01'
                    AND '2024-12-31'
  AND phase = 'PHASE3'
ORDER BY start_date DESC
LIMIT 3;

**LIKE — 글자가 들어간 행 찾기** — `%` 는 아무 글자가 몇 개든 온다는 뜻입니다. `LIKE` 는 대소문자를 구분하므로 `ILIKE` 도 함께 기억하세요 (5절 실수 4).

In [ ]:
%%sql
-- “제목에 Nurs 가 들어간 시험은 몇 건이야?”  → 187
SELECT COUNT(*) AS 간호관련
FROM studies
WHERE brief_title LIKE '%Nurs%';

**IS NULL — 값이 없는 행 찾기** — 완료된 19,622건 중 16,098건(82%)에 결과가 없습니다. `= NULL` 로 쓰면 오류 없이 0이 나오니 주의하세요 (5절 실수 1).

In [ ]:
%%sql
-- “완료됐는데 결과가 게시되지 않은 시험은?”
SELECT COUNT(*) AS 결과없음
FROM studies
WHERE overall_status = 'COMPLETED'
  AND results_first_posted_date IS NULL;

**여기까지 정리**
1. `SELECT` · `FROM` = 어떤 열을, 어느 표에서
2. `WHERE` = 어떤 행만. 문자 · 날짜는 '작은따옴표'
3. `IS NULL` 로 빈 값을 찾는다 (`= NULL` 은 안 됨)

**잠깐 생각해 봅시다** — 2024년에 시작한 시험만 보려면 어떤 절을 쓸까요?
`WHERE start_date >= '2024-01-01' AND start_date <= '2024-12-31'` 또는 `BETWEEN` 을 씁니다.

In [ ]:
%%sql
-- “2024년에 시작한 시험은 몇 건이야?”
SELECT COUNT(*) AS 시험수
FROM studies
WHERE start_date >= '2024-01-01' AND start_date <= '2024-12-31';

## 4. 정렬과 중복 제거 — ORDER BY · DISTINCT

**ORDER BY — 정렬** — 큰 값부터, 또는 최근 순으로 정렬합니다. `DESC` 는 내림차순(큰 값부터)입니다.
1위 188,814,085명(약 1억 8천만 명)은 실제 등록 인원일 수 없습니다. 정렬은 이런 이상한 값을 찾는 데도 씁니다 (12주차).

In [ ]:
%%sql
-- “등록인원이 가장 많은 시험 3건을 보여 줘”
SELECT nct_id, enrollment
FROM studies
ORDER BY enrollment DESC
LIMIT 3;

**DISTINCT — 값의 종류 보기** — WHERE 조건을 쓰기 전에 DISTINCT 로 값의 종류를 먼저 보면 오타와 빈 결과를 막을 수 있습니다.

In [ ]:
%%sql
-- “진행 상태에 어떤 값들이 있어?”  → 모두 9가지
SELECT DISTINCT overall_status
FROM studies
ORDER BY overall_status;

## 5. 자주 하는 실수 — 오류 메시지 읽기

`[오류 예시]` 셀은 **일부러 틀리게** 적었습니다. 실행해서 결과나 오류 메시지를 본 뒤, 바로 아래 고친 셀과 비교하세요. 오류는 잘못이 아니라 안내문입니다.
오류 메시지는 붉은 글씨 중 **`Error:` 가 들어간 줄**만 읽으면 됩니다 (이 절 끝의 ‘오류 메시지 읽는 법’).

### 실수 1 — `= NULL` (오류가 나지 않아 더 위험합니다)
NULL은 ‘같다/다르다’로 비교할 수 없어서, 오류 없이 0이 나옵니다. `IS NULL` 을 씁니다. 반대는 `IS NOT NULL` 입니다.

In [ ]:
%%sql
-- [오류 예시] 이렇게 쓰면 → 오류 없이 0
SELECT COUNT(*) AS 결과없음
FROM studies
WHERE overall_status = 'COMPLETED'
  AND results_first_posted_date = NULL;

In [ ]:
%%sql
-- 이렇게 고칩니다 → IS NULL (16,098)
SELECT COUNT(*) AS 결과없음
FROM studies
WHERE overall_status = 'COMPLETED'
  AND results_first_posted_date IS NULL;

결과가 0이면 “없다”가 아니라 **“조건이 틀렸다”** 를 먼저 의심하세요.

### 실수 2 — 작은따옴표 빠뜨림 (실행하면 오류 메시지가 나옵니다)
따옴표가 없으면 `COMPLETED` 를 열 이름으로 읽습니다 → `Binder Error: Referenced column "COMPLETED" not found`.
문자 값은 '작은따옴표'로 감쌉니다. 큰따옴표는 열 이름을 뜻합니다.

In [ ]:
%%sql
-- [오류 예시] 이렇게 쓰면 → Binder Error
SELECT COUNT(*) AS 완료시험
FROM studies
WHERE overall_status = COMPLETED;

In [ ]:
%%sql
-- 이렇게 고칩니다 → 'COMPLETED'
SELECT COUNT(*) AS 완료시험
FROM studies
WHERE overall_status = 'COMPLETED';

열 이름을 잘못 쓴 거라면 오류 메시지의 `Candidate bindings` 목록이 비슷한 열 이름을 알려 줍니다. 값을 쓰려던 거라면 작은따옴표를 확인하세요.

### 실수 3 — 절의 순서를 바꿈 (실행하면 오류 메시지가 나옵니다)
`LIMIT` 를 `ORDER BY` 앞에 쓰면 `Parser Error: syntax error at or near "ORDER"` 가 납니다.
절의 순서는 외우는 것이 아니라 “무엇을 · 어디서 · 어떤 행만 · 어떤 순서로 · 몇 줄”로 읽으면 됩니다.

In [ ]:
%%sql
-- [오류 예시] 이렇게 쓰면 → Parser Error
SELECT nct_id
FROM studies
LIMIT 5
ORDER BY start_date

In [ ]:
%%sql
-- 이렇게 고칩니다 → SELECT, FROM, WHERE, ORDER BY, LIMIT 순서
SELECT nct_id
FROM studies
ORDER BY start_date
LIMIT 5

### 실수 4 — 대소문자 (오류가 나지 않아 더 위험합니다)
`LIKE` 는 대소문자를 구분합니다. 소문자 `covid` 로 찾으면 6건만 나옵니다. 대소문자를 가리지 않고 세면 2,125건입니다.

In [ ]:
%%sql
-- [오류 예시] 이렇게 쓰면 → 6건만 (오류는 없음)
SELECT COUNT(*) AS 시험수
FROM studies
WHERE brief_title LIKE '%covid%';

In [ ]:
%%sql
-- 이렇게 고칩니다 → ILIKE
SELECT COUNT(*) AS 시험수
FROM studies
WHERE brief_title ILIKE '%covid%';

`ILIKE` 는 대소문자를 구분하지 않습니다. `UPPER()` 를 써도 됩니다: `UPPER(brief_title)` 는 제목을 모두 대문자로 바꾸므로, 찾는 글자도 대문자로 적습니다 (Q5에서 연습합니다).
조건을 쓰기 전에 DISTINCT 로 실제 값의 모양을 먼저 확인하세요.

> 참고: `ILIKE` 는 DuckDB · PostgreSQL 에서 쓰는 문법입니다. 어느 DBMS에서나 되는 방법은 `UPPER(...) LIKE` 입니다.

### 오류 메시지 읽는 법
1. 붉은 글씨가 길게 나와도 **전부 읽지 않아도 됩니다.**
2. **`Error:` 가 들어간 줄**(예: `Binder Error: …`, `Parser Error: …`)이 진짜 원인입니다.
3. 자주 보는 세 가지

| 이런 줄이 보이면 | 뜻 |
|---|---|
| `Binder Error: Referenced column "…" not found` | 열 이름 오타 또는 따옴표 누락 |
| `Parser Error: syntax error at or near "…"` | 절의 순서나 쉼표 문제 |
| `Conversion Error` / `Cannot compare` | 타입이 맞지 않음 (5주차) |

4. **오류 없이 결과만 이상할 때가 가장 위험합니다.** 건수를 확인하세요.

## 직접 해보기 — 8문항

각 문항의 `%%sql` 셀에서 `___` 를 채워 실행한 뒤, **바로 다음 셀**의 `check(...)` 를 실행하세요. 두 번 틀리면 힌트가 나옵니다.
값의 철자(대문자 · 소문자)는 문항에 적힌 그대로 씁니다.

**Q1.** SELECT · AS — “임상시험의 등록번호(`nct_id`)와 진행 상태(`overall_status`)를 `등록번호`, `진행상태` 라는 이름으로 보여 줘”

모든 시험(37,509행)이 나오면 됩니다.

In [ ]:
%%sql
-- Q1
SELECT ___ AS 등록번호,
       ___ AS 진행상태
FROM studies;

In [ ]:
check("Q1", _)

**Q2.** WHERE = — “진행 상태(`overall_status`)가 중단(`TERMINATED`)인 시험은 몇 건이야?”

In [ ]:
%%sql
-- Q2
SELECT COUNT(*) AS 중단시험
FROM studies
WHERE ___;

In [ ]:
check("Q2", _)

**Q3.** AND · 비교 — “등록인원(`enrollment`)이 1000명 이상이면서 완료(`COMPLETED`)된 시험은 몇 건이야?”

In [ ]:
%%sql
-- Q3
SELECT COUNT(*) AS 시험수
FROM studies
WHERE ___
  AND ___;

In [ ]:
check("Q3", _)

**Q4.** BETWEEN — “2020년에 시작한(`start_date`) 시험은 몇 건이야?”

In [ ]:
%%sql
-- Q4
SELECT COUNT(*) AS 시험수
FROM studies
WHERE start_date BETWEEN ___ AND ___;

In [ ]:
check("Q4", _)

**Q5.** LIKE — “제목(`brief_title`)에 sleep(수면)이 들어간 시험은 몇 건이야? Sleep, SLEEP 처럼 대소문자가 달라도 모두 세어 줘”

`UPPER(brief_title)` 는 제목을 모두 대문자로 바꿉니다 (5절 실수 4).

In [ ]:
%%sql
-- Q5
SELECT COUNT(*) AS 시험수
FROM studies
WHERE UPPER(brief_title) LIKE ___;

In [ ]:
check("Q5", _)

**Q6.** IS NOT NULL — “완료(`COMPLETED`)됐고 결과가 게시된(`results_first_posted_date` 에 값이 있는) 시험은 몇 건이야?”

In [ ]:
%%sql
-- Q6
SELECT COUNT(*) AS 결과있음
FROM studies
WHERE overall_status = 'COMPLETED'
  AND ___;

In [ ]:
check("Q6", _)

> 확인: Q6의 결과있음 + 3절의 결과없음(16,098) = 완료시험(19,622) 이 되나요? 이렇게 건수를 맞춰 보는 습관이 중요합니다.

**Q7.** ORDER BY · LIMIT — “3상(`PHASE3`) 시험 중 등록인원(`enrollment`)이 가장 많은 5건을 많은 순으로 보여 줘”

In [ ]:
%%sql
-- Q7
SELECT nct_id, enrollment
FROM studies
WHERE phase = 'PHASE3'
ORDER BY ___
LIMIT ___;

In [ ]:
check("Q7", _)

**Q8.** DISTINCT — “시험 단계(`phase`)에는 어떤 값들이 있어?” 값의 종류만 중복 없이 보세요.

In [ ]:
%%sql
-- Q8
SELECT ___ phase
FROM studies;

In [ ]:
check("Q8", _)

> 결과의 `NA` 는 ‘해당 없음’이라는 글자 값이고, `None` 은 값이 비어 있는(NULL) 것입니다. 둘 다 한 가지 종류로 셉니다.

## 오늘 배운 것

- DDL로 표를 만들고, DML로 꺼내고, DCL로 권한을 정한다
- **SELECT → FROM → WHERE → ORDER BY → LIMIT** 순서로 적는다
- 문자 · 날짜는 '작은따옴표', 빈 값은 **IS NULL** 로 찾는다
- 조건을 하나씩 더하며 **건수가 어떻게 변하는지** 확인한다

다음 시간: 8강. SQL (2) — 집계(COUNT · AVG)와 GROUP BY, 그리고 JOIN

아래 셀을 실행하면 8문항의 결과를 한눈에 볼 수 있습니다.

In [ ]:
진행확인()